# YTAgent — Google Colab Free GPU Worker

This notebook connects your free Google Colab GPU to your YTAgent app. It does **not** render a separate local film or require you to upload clips manually. It waits for a YTAgent video job, generates ten Wan 2.1 scenes and ten Kokoro narration tracks, then securely uploads them to Railway. YTAgent assembles the finished MP4 in the dashboard.

## Before running
1. Open **Runtime → Change runtime type → T4 GPU** (or another free GPU if offered).
2. In Colab's **Secrets** panel (key icon), add `YT_AGENT_WORKER_TOKEN`. Use the same private value already configured in the YTAgent Railway service, and enable notebook access.
3. Run cells from top to bottom. Keep this Colab session open while a film is generating.

Google's free GPU availability and session duration are not guaranteed. Do not buy compute units for this project. The notebook uses public model weights and does not need a Hugging Face token.

In [ ]:
# 1. Get the current YTAgent worker code and install runtime dependencies.
import os, pathlib, subprocess, sys, json, urllib.request, time
REPO = "/content/YTAgent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/qingremzy/YTAgent.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO + "/gpu-worker")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")
def run(cmd):
    print("RUN:", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True)
run(["apt-get", "-qq", "update"])
run(["apt-get", "-qq", "install", "-y", "ffmpeg", "espeak-ng", "fonts-dejavu-core"])
# Keep Colab's preinstalled CUDA-enabled PyTorch. Do not pip-install or replace torch.
run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.32.0", "transformers>=4.45.0", "accelerate>=1.0.0", "ftfy", "imageio[ffmpeg]", "soundfile>=0.12.1", "fastapi>=0.115.0", "pydantic>=2.0", "spacy>=3.8.2,<4", "num2words", "phonemizer-fork", "espeakng-loader", "loguru", "addict"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "git+https://github.com/hexgrad/misaki.git"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "git+https://github.com/hexgrad/kokoro.git"])
print("Worker dependencies installed.")

In [ ]:
# 2. Check that Colab assigned a GPU and securely connect to YTAgent.
import torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU was assigned. Choose Runtime > Change runtime type > GPU, reconnect, and rerun this cell.")
torch.cuda.init()
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
from google.colab import userdata
try:
    worker_token = userdata.get("YT_AGENT_WORKER_TOKEN")
except Exception as exc:
    raise RuntimeError("Add YT_AGENT_WORKER_TOKEN in Colab Secrets and enable notebook access.") from exc
if not worker_token:
    raise RuntimeError("YT_AGENT_WORKER_TOKEN is empty. Add the same private token configured in Railway to Colab Secrets.")
os.environ["YT_AGENT_WORKER_TOKEN"] = worker_token
os.environ["YT_AGENT_SERVER_URL"] = "https://ytagent-production-1038.up.railway.app"
request = urllib.request.Request(os.environ["YT_AGENT_SERVER_URL"] + "/api/worker/heartbeat", headers={"Authorization": "Bearer " + worker_token})
with urllib.request.urlopen(request, timeout=20) as response:
    health = json.loads(response.read().decode("utf-8"))
if not health.get("workerOnline"):
    raise RuntimeError("YTAgent did not confirm the worker heartbeat.")
print("Connected securely to YTAgent. GPU worker is online and ready.")

In [ ]:
# 3. Wait for one YTAgent job, generate it, and upload scenes + narration automatically.
print("YTAgent is waiting for a video job. Submit a film topic in the YTAgent dashboard while this cell is running.", flush=True)
print("This worker waits up to 20 minutes for one job. If no job arrives, rerun this cell.", flush=True)
subprocess.run([sys.executable, "-u", "app.py", "--run-once", "--wait-seconds", "1200"], check=True)
print("The one-job worker has finished. Check YTAgent for the final MP4 and generation status.")

## How to use it

- Start this notebook first and leave cell 3 running.
- In another tab, open [YTAgent](https://ytagent-production-1038.up.railway.app) and submit a topic.
- The Colab worker claims the job, creates the moving scenes and narration, and uploads them over authenticated HTTPS.
- YTAgent compiles and quality-checks one complete vertical MP4. Download it from YTAgent when it is ready.

The notebook processes one film per run. Rerun cell 3 for the next film. If Colab disconnects or free GPU capacity is unavailable, reconnect and rerun the cells. No Cloudflare tunnel, public worker URL, paid video API, or manual clip stitching is used.